# Local RAG with Ollama + Local Embeddings (Sentence-Transformers) + ChromaDB

This notebook demonstrates a **fully local** Retrieval-Augmented Generation (RAG) pipeline:

- **LLM**: Ollama (local server, open-source model)
- **Embeddings**: `sentence-transformers` (runs on CPU)
- **Vector DB**: ChromaDB (local, lightweight)

## Requirements
- Ollama installed and running
- A model pulled, e.g.:
```bash
ollama pull llama3.2:3b
```

> **If you are running this inside Docker Compose** (provided in this tutorial), Ollama is reachable at `http://ollama:11434`.  
> If you are running it on your host machine, use `http://localhost:11434`.

In [1]:
import os
import requests

# Resolve Ollama endpoint depending on environment
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
print('Using Ollama at:', BASE_URL)

# Diagnostic check
try:
    r = requests.get(BASE_URL, timeout=3)
    print('Server reachable. Status:', r.status_code)
    try:
        tags = requests.get(f"{BASE_URL}/api/tags", timeout=3)
        print('Tags endpoint status:', tags.status_code)
    except Exception as e:
        print('Tags endpoint check failed:', e)
except Exception as e:
    print('Server not reachable:', e)


Using Ollama at: http://host.docker.internal:11434
Server reachable. Status: 200
Tags endpoint status: 200


In [2]:
import os

# Switch automatically depending on environment:
# - In Docker Compose: OLLAMA_BASE_URL defaults to http://ollama:11434
# - On host: set OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")

print("OLLAMA_BASE_URL =", OLLAMA_BASE_URL)
print("MODEL =", MODEL)


OLLAMA_BASE_URL = http://host.docker.internal:11434
MODEL = llama3.2:3b


## 1) Health check: is Ollama reachable?

In [3]:
import requests

r = requests.get(OLLAMA_BASE_URL, timeout=10)
print("Status:", r.status_code)
print("Body:", r.text[:200])


Status: 200
Body: Ollama is running


## 2) Install / import libraries
If you're in the provided Docker image, these should already be available.

In [4]:
# If you are NOT using the provided Docker image, you can install dependencies with:
# !pip install -U chromadb sentence-transformers requests

import chromadb
from sentence_transformers import SentenceTransformer

2026-03-05 12:58:23.633623144 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3) Build a tiny knowledge base (you can replace this with your own documents)

We will embed each chunk and store it in ChromaDB.

In [5]:
documents = [
    {
        "id": "doc1",
        "text": "Ollama is a local runtime that runs open-source LLMs and exposes a REST API on port 11434.",
        "source": "lecture_notes"
    },
    {
        "id": "doc2",
        "text": "Quantization reduces model size (e.g., 4-bit, 8-bit) to make inference faster and feasible on CPUs.",
        "source": "lecture_notes"
    },
    {
        "id": "doc3",
        "text": "RAG (Retrieval-Augmented Generation) combines retrieval from a knowledge base with an LLM to answer questions grounded in documents.",
        "source": "lecture_notes"
    },
    {
        "id": "doc4",
        "text": "ChromaDB is a lightweight vector database for storing embeddings and performing similarity search locally.",
        "source": "lecture_notes"
    },
    {
        "id": "doc5",
        "text": "Ollama provides endpoints like /api/generate for single-turn generation and /api/chat for multi-turn conversations.",
        "source": "lecture_notes"
    },
]

## 4) Create embeddings locally (CPU)

We use a small, fast sentence-transformer model.  
The first run may download weights (unless already bundled in the Docker image).

In [6]:
EMBED_MODEL_NAME = os.getenv("EMBED_MODEL_NAME", "sentence-transformers/all-MiniLM-L6-v2")
embedder = SentenceTransformer(EMBED_MODEL_NAME)

texts = [d["text"] for d in documents]
embeddings = embedder.encode(texts, show_progress_bar=False).tolist()

len(embeddings), len(embeddings[0])

(5, 384)

## 5) Store embeddings in ChromaDB

We store:
- `ids`
- `documents` (text)
- `metadatas`
- `embeddings`

In [7]:
client = chromadb.Client()
collection = client.get_or_create_collection(name="ollama_rag_demo")

# Clean for repeatable runs
existing = collection.get()
if existing and existing.get("ids"):
    collection.delete(ids=existing["ids"])

collection.add(
    ids=[d["id"] for d in documents],
    documents=[d["text"] for d in documents],
    metadatas=[{"source": d["source"]} for d in documents],
    embeddings=embeddings
)

print("Stored:", collection.count(), "documents")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored: 5 documents


## 6) Retrieval function (top-k similarity search)

In [8]:
def retrieve(query: str, k: int = 3):
    q_emb = embedder.encode([query], show_progress_bar=False).tolist()
    results = collection.query(
        query_embeddings=q_emb,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    hits = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        hits.append({"text": doc, "meta": meta, "distance": dist})
    return hits

hits = retrieve("What is RAG?", k=3)
hits

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[{'text': 'RAG (Retrieval-Augmented Generation) combines retrieval from a knowledge base with an LLM to answer questions grounded in documents.',
  'meta': {'source': 'lecture_notes'},
  'distance': 1.1129646301269531},
 {'text': 'Ollama is a local runtime that runs open-source LLMs and exposes a REST API on port 11434.',
  'meta': {'source': 'lecture_notes'},
  'distance': 1.7079542875289917},
 {'text': 'ChromaDB is a lightweight vector database for storing embeddings and performing similarity search locally.',
  'meta': {'source': 'lecture_notes'},
  'distance': 1.7192105054855347}]

## 7) Ask the LLM with retrieved context (RAG)

We build a prompt with:
- retrieved context
- a question
- instructions to cite the context

This is a simple *prompt-based* grounding approach.

In [9]:
def ollama_generate(prompt: str, model: str = MODEL):
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {"model": model, "prompt": prompt, "stream": False}
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["response"]

def rag_answer(question: str, k: int = 3):
    
    hits = retrieve(question, k=k)

    context = "\n\n".join(
        [f"[{i+1}] {h['text']}" for i, h in enumerate(hits)]
    )

    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the context below.
If the answer is not present, say:
"I don't know based on the provided context."

Context:
{context}

Question: {question}

Answer (include references like [1], [2] when relevant):
"""

    answer = ollama_generate(prompt)

    return answer, hits


answer, hits = rag_answer(
    "Which Ollama endpoints are available for chat and generation?",
    k=3
)

print(answer)


According to the context, for chat and generation, the following Ollama endpoints are available:

- /api/chat for multi-turn conversations
- /api/generate for single-turn generation

These endpoints are exposed by the Ollama REST API on port 11434, as mentioned in [2].


## 8) Mini-lab: try your own questions

Suggestions:
- "What is quantization and why do we use it?"
- "What port does Ollama use?"
- "Explain RAG in one paragraph."

In [10]:
question = "What is quantization and why is it useful on CPUs?"
answer, hits = rag_answer(question, k=3)
print(answer)

According to the provided context, quantization is the process of reducing model size (e.g., 4-bit, 8-bit) to make inference faster and more feasible on CPUs ([1]). This is done to improve the performance of models on low-power computing devices, such as CPUs.


## 9) Next steps (optional extensions)

- Replace the toy `documents` list with:
  - lecture notes
  - PDFs (text extraction)
  - a course wiki or KB
- Add chunking (split long documents into smaller pieces)
- Add re-ranking (improve retrieval quality)
- Store the ChromaDB collection on disk (persistence)